# 07 | Does satellite information improve upon seasonality?
The original seasonal baseline beat all satellite-only models. We now ask whether SAR and optical features explain **departures from that baseline**, and whether they distinguish locations within a given month. This is a post-hoc exploratory extension: the original held-out scores have already been inspected. Keep all original outputs unchanged. No new downloads or GPU are required.

## Setup (CPU, standard RAM)
Run this cell first in each notebook. The project uses small monthly raster tiles and tabular ML. If Colab requests a restart, use Runtime → Restart session and rerun from the beginning. Do not change NumPy manually. Files persist in Drive.

In [ ]:
%pip -q install earthengine-api rasterio pyproj scikit-learn joblib
import sys,json
from pathlib import Path
from importlib.metadata import version
for pkg,mod in [('numpy','numpy'),('scikit-learn','sklearn'),('rasterio','rasterio')]:
    loaded=sys.modules.get(mod)
    if loaded is not None and getattr(loaded,'__version__',version(pkg))!=version(pkg):
        raise RuntimeError(f'{pkg} changed. Restart session and rerun.')
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Bagre_SAR_Soil_Moisture')
assert (ROOT/'code'/'sar_workflow.py').is_file(), 'Check Drive folder and shared code.'
sys.path.insert(0,str(ROOT/'code'))
import sar_workflow as sw
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
for sub in ['data','models','outputs','figures','qgis']: (ROOT/sub).mkdir(exist_ok=True)
print('NumPy:',version('numpy'),'scikit-learn:',version('scikit-learn'))

## Fit only on the original training rows
First fit the seasonal harmonic using the original ridge settings. Its residual is observed SMAP minus seasonal prediction. Fit fixed-penalty ridge models to these residuals using SAR, NDVI or both. Separately compare direct additive models with seasonal and satellite predictors. No test targets enter fitting or model selection. A residual here is not a long-term climatological anomaly: only one training year is available.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
df=sw.add_season(pd.read_csv(ROOT/'data'/'coarse_monthly_table.csv'))
df=df[df.eligible].copy()
train=df[df.split=='train'].copy()
season=['sin_month','cos_month']
def ridge(): return make_pipeline(StandardScaler(),Ridge(alpha=5))
base=ridge().fit(train[season],train.sm)
df['seasonal_prediction']=base.predict(df[season])
df['observed_anomaly']=df.sm-df.seasonal_prediction
train=df[df.split=='train'].copy()
models={}; predictions={'seasonal_ridge':df.seasonal_prediction.to_numpy()}
for name,cols in [('sar',sw.FEATURES[:3]),('optical',['ndvi']),('fusion',sw.FEATURES)]:
    residual=ridge().fit(train[cols],train.observed_anomaly)
    additive=ridge().fit(train[season+cols],train.sm)
    models['residual_'+name]=(cols,residual)
    models['seasonal_'+name]=(season+cols,additive)
    predictions['residual_'+name]=df.seasonal_prediction.to_numpy()+residual.predict(df[cols])
    predictions['seasonal_'+name]=additive.predict(df[season+cols])
print('Training rows:',len(train)); display(df.groupby('split').size())

## Compare on identical held-out rows
Positive skill means lower squared error than the seasonal baseline (1 − MSE_model/MSE_seasonal). A negative value means satellite features made predictions worse. Compare both absolute soil moisture and deviations from the seasonal baseline. Model choice uses validation RMSE only; all candidates are reported.

In [ ]:
splits=['validation','spatial_test','temporal_test','joint_test']
rows=[]; out=[]
for name,p in predictions.items():
    z=df.copy(); z['model']=name; z['prediction']=p; z['predicted_anomaly']=p-z.seasonal_prediction
    for split in splits:
        d=z[z.split==split].copy(); m=sw.metrics(d.sm,d.prediction)
        b=np.mean((d.sm-d.seasonal_prediction)**2)
        m['skill_vs_seasonal']=1-m['rmse']**2/b if b>0 else np.nan
        m['anomaly_r']=sw.metrics(d.observed_anomaly,d.predicted_anomaly)['r'] if name!='seasonal_ridge' else np.nan
        rows.append(dict(model=name,split=split,**m)); out.append(d)
metrics=pd.DataFrame(rows); pred=pd.concat(out,ignore_index=True)
selected=metrics[metrics.split=='validation'].sort_values('rmse').iloc[0].model
print('Validation-selected exploratory model:',selected)
display(metrics.round(4)); metrics.to_csv(ROOT/'outputs'/'07_anomaly_metrics.csv',index=False)
pred.to_csv(ROOT/'outputs'/'07_anomaly_predictions.csv',index=False)

## Within-month spatial skill
Center observations and predictions separately within each held-out month and split. This is an evaluation-only diagnostic, not a transformation used for prediction. It removes the shared seasonal mean. A constant seasonal prediction has zero spatial variation, so its within-month correlation is undefined. Positive centered skill indicates improvement over predicting no spatial contrast.

In [ ]:
rows=[]
for (name,split),d in pred.groupby(['model','split']):
    d=d.copy(); d=d[d.groupby('month').month.transform('size')>=2]
    y=d.sm-d.groupby('month').sm.transform('mean')
    p=d.prediction-d.groupby('month').prediction.transform('mean')
    m=sw.metrics(y,p); baseline=np.mean(y**2)
    m['spatial_skill_vs_zero']=1-m['rmse']**2/baseline if baseline>0 else np.nan
    if name=='seasonal_ridge': m['r']=np.nan
    rows.append(dict(model=name,split=split,months=d.month.nunique(),**m))
within=pd.DataFrame(rows); display(within.round(4)); within.to_csv(ROOT/'outputs'/'07_within_month_metrics.csv',index=False)

## Blog-ready comparisons
Use common axes and label physical units. Compare skill and within-month performance rather than relying on R² alone. Save both high-resolution PNG and vector PDF.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5),constrained_layout=True)
order=list(predictions); colors=['#64748b','#2563eb','#0d9488','#7c3aed','#60a5fa','#5eead4','#c4b5fd']
for ax,tab,col,title in [(axes[0],metrics,'skill_vs_seasonal','Total skill vs seasonal baseline'),(axes[1],within,'spatial_skill_vs_zero','Within-month spatial skill')]:
    piv=tab.pivot(index='split',columns='model',values=col).reindex(index=splits,columns=order)
    piv.plot.bar(ax=ax,color=colors,legend=False); ax.axhline(0,color='black',lw=1); ax.set(title=title,ylabel='Relative squared-error skill',xlabel=''); ax.tick_params(axis='x',rotation=20); ax.grid(axis='y',alpha=.2)
handles,labels=axes[0].get_legend_handles_labels(); fig.legend(handles,labels,loc='outside lower center',ncol=4,fontsize=8)
for ext in ['png','pdf']: fig.savefig(ROOT/'figures'/f'07_anomaly_skill.{ext}',dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(12,4),constrained_layout=True)
for ax,split in zip(axes,splits[1:]):
    d=pred[(pred.model==selected)&(pred.split==split)]
    ax.scatter(d.observed_anomaly,d.predicted_anomaly,s=10,alpha=.5,color='#167d8d')
    lim=max(.02,np.max(np.abs(d[['observed_anomaly','predicted_anomaly']].to_numpy())))
    ax.plot([-lim,lim],[-lim,lim],'k--'); ax.set(xlim=(-lim,lim),ylim=(-lim,lim),xlabel='Observed residual (m³/m³)',ylabel='Predicted residual (m³/m³)',title=f'{split} (n={len(d)})'); ax.set_aspect('equal')
fig.suptitle(f'Exploratory anomaly retrieval: {selected}')
for ext in ['png','pdf']: fig.savefig(ROOT/'figures'/f'07_anomaly_scatter.{ext}',dpi=300,bbox_inches='tight')
plt.show()

**Final checkpoint:** report whether the validation-selected extension improves on the original seasonal baseline on all held-out splits and whether spatial contrast skill is positive. A negative result is informative. Do not interpret the relative 250 m map as validated soil moisture. A genuinely independent future year or field measurements are needed to confirm the exploratory extension.